In [0]:
bronze_path = """
abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/bronze/state_elections/state=nrw/nrw2022.csv
""".strip()
df_nrw2022 =( 
spark.read
.option("header", "true")
.option("inferSchema", "true")
.option("skipRows", "4")
.option("sep", ";")
.csv(bronze_path)
)
df_nrw2022.printSchema()

root
 |-- Wahl: string (nullable = true)
 |-- Wahlkreisnr.: integer (nullable = true)
 |-- Wahlkreisname: string (nullable = true)
 |-- A1: integer (nullable = true)
 |-- A2: integer (nullable = true)
 |-- A3: integer (nullable = true)
 |-- A: integer (nullable = true)
 |-- B: integer (nullable = true)
 |-- B1: integer (nullable = true)
 |-- C: integer (nullable = true)
 |-- D: integer (nullable = true)
 |-- D1: integer (nullable = true)
 |-- D2: integer (nullable = true)
 |-- D3: integer (nullable = true)
 |-- D4: integer (nullable = true)
 |-- D5: integer (nullable = true)
 |-- D6: integer (nullable = true)
 |-- D7: integer (nullable = true)
 |-- D8: integer (nullable = true)
 |-- D9: integer (nullable = true)
 |-- D10: string (nullable = true)
 |-- D11: string (nullable = true)
 |-- D12: integer (nullable = true)
 |-- D13: integer (nullable = true)
 |-- D14: integer (nullable = true)
 |-- D15: string (nullable = true)
 |-- D16: integer (nullable = true)
 |-- D17: integer (nullable =

In [0]:
d_cols = [c for c in df_nrw2022.columns if c.startswith("D")]
f_cols = [c for c in df_nrw2022.columns if c.startswith("F")]

print("D-Spalten:", len(d_cols))
print("F-Spalten:", len(f_cols))
print(d_cols)
print(f_cols)


D-Spalten: 40
F-Spalten: 40
['D', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D21', 'D22', 'D23', 'D24', 'D25', 'D26', 'D27', 'D28', 'D29', 'D30', 'D31', 'D32', 'D33', 'D34', 'D35', 'D36', 'D37', 'D38', 'D39']
['F', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29', 'F30', 'F31', 'F32', 'F33', 'F34', 'F35', 'F36', 'F37', 'F38', 'F39']


In [0]:
df_nrw2022 = df_nrw2022.withColumnRenamed(
    "Wahlkreisnr.",
    "wahlkreisnr"
)

In [0]:
df_nrw2022.select(
    "Wahl",
    "Wahlkreisnr",
    "Wahlkreisname",
    "D1",
    "D2",
    "F1",
    "F2"
).show(10, truncate=False)

+----+-----------+--------------------+-------+-------+-------+-------+
|Wahl|Wahlkreisnr|Wahlkreisname       |D1     |D2     |F1     |F2     |
+----+-----------+--------------------+-------+-------+-------+-------+
|LW22|0          |Nordrhein-Westfalen |2607596|2092933|2552276|1905002|
|LW22|1          |Aachen I            |13624  |10945  |13243  |9838   |
|LW22|2          |Aachen II           |15948  |11788  |15085  |10848  |
|LW22|3          |Aachen III          |23105  |19208  |22566  |18084  |
|LW22|4          |Aachen IV           |22195  |21231  |23462  |17418  |
|LW22|5          |Rhein-Erft-Kreis I  |30282  |18184  |27428  |17260  |
|LW22|6          |Rhein-Erft-Kreis II |23798  |17104  |22677  |16136  |
|LW22|7          |Rhein-Erft-Kreis III|23645  |16260  |23603  |15968  |
|LW22|8          |Euskirchen I        |25902  |17948  |27066  |14723  |
|LW22|9          |Heinsberg I         |23583  |11355  |23149  |10840  |
+----+-----------+--------------------+-------+-------+-------+-

In [0]:
party_mapping_2022 = {
    1: "CDU",
    2: "SPD",
    3: "FDP",
    4: "AfD",
    5: "GRÜNE",
    6: "DIE LINKE",
    7: "PIRATEN",
    8: "Die PARTEI",
    9: "FREIE WÄHLER",
    10: "BIG",
    11: "ÖDP",
    12: "Volksabstimmung",
    13: "MLPD",
    14: "DIE VIOLETTEN",
    15: "Gesundheitsforschung",
    16: "ZENTRUM",
    17: "DKP",
    18: "dieBasis",
    19: "DSP",
    20: "Die Urbane.",
    21: "LIEBE",
    22: "FAMILIE",
    23: "neo",
    24: "Die Humanisten",
    25: "PdF",
    26: "LfK",
    27: "Tierschutzpartei",
    28: "Team Todenhöfer",
    29: "Volt"
}
# Erststimmen: D1 bis D39
d_vote_cols = [f"D{i}" for i in range(1, 40)]

# Zweitstimmen: nur F1 bis F29
# F30-F39 sind keine normalen Landeslisten-Stimmen
f_vote_cols = [f"F{i}" for i in range(1, 30)]



In [0]:
from pyspark.sql import functions as F

vote_cols_2022 = d_vote_cols + f_vote_cols

for c in vote_cols_2022:
    df_nrw2022 = df_nrw2022.withColumn(
        c,
        F.when(
            F.trim(F.col(c).cast("string")).isin("", "-", "x"),
            None
        ).otherwise(
            F.col(c).cast("long")
        )
    )

In [0]:
df_nrw2022.select(
    *d_vote_cols[:5],
    *f_vote_cols[:5]
).printSchema()

root
 |-- D1: long (nullable = true)
 |-- D2: long (nullable = true)
 |-- D3: long (nullable = true)
 |-- D4: long (nullable = true)
 |-- D5: long (nullable = true)
 |-- F1: long (nullable = true)
 |-- F2: long (nullable = true)
 |-- F3: long (nullable = true)
 |-- F4: long (nullable = true)
 |-- F5: long (nullable = true)



In [0]:
long_zweit = []

for i in range(1, 30):
    temp = (
        df_nrw2022
        .select(
            "Wahl",
            "wahlkreisnr",
            "Wahlkreisname",
            F.lit(party_mapping_2022[i]).alias("wahlvorschlag"),
            F.col(f"F{i}").alias("stimmen")
        )
        .withColumn("stimmenart", F.lit("Zweitstimme"))
    )

    long_zweit.append(temp)

In [0]:
df_zweit_2022 = long_zweit[0]

for temp in long_zweit[1:]:
    df_zweit_2022 = df_zweit_2022.unionByName(temp)

In [0]:
df_zweit_2022.show(40, truncate=False)

+----+-----------+--------------------------------------+-------------+-------+-----------+
|Wahl|wahlkreisnr|Wahlkreisname                         |wahlvorschlag|stimmen|stimmenart |
+----+-----------+--------------------------------------+-------------+-------+-----------+
|LW22|0          |Nordrhein-Westfalen                   |CDU          |2552276|Zweitstimme|
|LW22|1          |Aachen I                              |CDU          |13243  |Zweitstimme|
|LW22|2          |Aachen II                             |CDU          |15085  |Zweitstimme|
|LW22|3          |Aachen III                            |CDU          |22566  |Zweitstimme|
|LW22|4          |Aachen IV                             |CDU          |23462  |Zweitstimme|
|LW22|5          |Rhein-Erft-Kreis I                    |CDU          |27428  |Zweitstimme|
|LW22|6          |Rhein-Erft-Kreis II                   |CDU          |22677  |Zweitstimme|
|LW22|7          |Rhein-Erft-Kreis III                  |CDU          |23603  |Z

In [0]:
# erstmal D1-D29 sicher mappen
party_mapping_erst_2022 = party_mapping_2022.copy()

long_erst = []

for i in range(1, 30):
    temp = (
        df_nrw2022
        .select(
            "Wahl",
            "wahlkreisnr",
            "Wahlkreisname",
            F.lit(party_mapping_erst_2022[i]).alias("wahlvorschlag"),
            F.col(f"D{i}").alias("stimmen")
        )
        .withColumn("stimmenart", F.lit("Erststimme"))
    )

    long_erst.append(temp)

df_erst_2022 = long_erst[0]

for temp in long_erst[1:]:
    df_erst_2022 = df_erst_2022.unionByName(temp)

In [0]:
df_long_2022 = df_erst_2022.unionByName(df_zweit_2022)

In [0]:
df_long_2022 = (
    df_long_2022
    .withColumn("wahljahr", F.lit(2022))
    .withColumn("wahltyp", F.lit("Landtagswahl"))
    .withColumn("bundesland", F.lit("Nordrhein-Westfalen"))
)

In [0]:
df_long_2022.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_long_2022.columns
]).show(truncate=False)

+----+-----------+-------------+-------------+-------+----------+--------+-------+----------+
|Wahl|wahlkreisnr|Wahlkreisname|wahlvorschlag|stimmen|stimmenart|wahljahr|wahltyp|bundesland|
+----+-----------+-------------+-------------+-------+----------+--------+-------+----------+
|0   |0          |0            |0            |2632   |0         |0       |0      |0         |
+----+-----------+-------------+-------------+-------+----------+--------+-------+----------+



In [0]:
rows = df_long_2022.count()
distinct_rows = df_long_2022.distinct().count()

print("Rows:", rows)
print("Distinct:", distinct_rows)
print("Duplicates:", rows - distinct_rows)

Rows: 7482
Distinct: 7482
Duplicates: 0


In [0]:
df_long_2022.groupBy("stimmenart").count().show()

+-----------+-----+
| stimmenart|count|
+-----------+-----+
|Zweitstimme| 3741|
| Erststimme| 3741|
+-----------+-----+



In [0]:
df_long_2022.select(
    "stimmenart",
    "wahlvorschlag"
).distinct().groupBy(
    "stimmenart"
).count().show()

+-----------+-----+
| stimmenart|count|
+-----------+-----+
| Erststimme|   29|
|Zweitstimme|   29|
+-----------+-----+



In [0]:
silver_path_nrw2022 = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "silver/state_elections/state=nrw/election_year=2022/"
)

df_long_2022.write.mode("overwrite").parquet(
    silver_path_nrw2022
)